In [ ]:
!pip install dash yfinance feedparser

In [ ]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objects as go
import yfinance as yf
import requests
import feedparser
import pandas as pd
import numpy as np
from datetime import datetime
from tensorflow.keras.models import load_model
from tensorflow.keras import backend as K
import joblib

In [ ]:
# 사용자 정의 F1 지표 함수 (모델 로드 시 필요)
def f1(y_true, y_pred):
    y_true = K.cast(y_true, 'float32')
    y_pred = K.cast(y_pred, 'float32')
    def recall(y_true, y_pred):
        true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
        possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
        return true_positives / (possible_positives + K.epsilon())
    def precision(y_true, y_pred):
        true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
        predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
        return true_positives / (predicted_positives + K.epsilon())
    precision_pos = precision(y_true, y_pred)
    recall_pos = recall(y_true, y_pred)
    precision_neg = precision((K.ones_like(y_true)-y_true), (K.ones_like(y_pred)-K.clip(y_pred, 0, 1)))
    recall_neg = recall((K.ones_like(y_true)-y_true), (K.ones_like(y_pred)-K.clip(y_pred, 0, 1)))
    f_posit = 2 * ((precision_pos * recall_pos) / (precision_pos + recall_pos + K.epsilon()))
    f_neg = 2 * ((precision_neg * recall_neg) / (precision_neg + recall_neg + K.epsilon()))
    return (f_posit + f_neg) / 2

In [ ]:
# 사전 학습된 모델과 스케일러 로드
model = load_model("nasdaq_cnn_lstm.keras", custom_objects={'f1': f1})
scaler = joblib.load("nasdaq_scaler.pkl")

app = dash.Dash(__name__)

US_TOP10 = {
    'NVDA': 'NVIDIA', 'AAPL': 'Apple', 'MSFT': 'Microsoft', 'AMZN': 'Amazon',
    'GOOGL': 'Alphabet', 'META': 'Meta', 'TSLA': 'Tesla',
    'AVGO': 'Broadcom', 'BRK-B': 'Berkshire Hathaway', 'LLY': 'Eli Lilly'
}
INDICES = {'^IXIC': 'NASDAQ', '^GSPC': 'S&P 500', '^DJI': 'Dow Jones'}

app.layout = html.Div(style={'backgroundColor': '#0b0f19', 'color': '#f8fafc', 'padding': '20px', 'fontFamily': 'sans-serif'}, children=[
    html.H1("📊 나스닥 통합 실시간 대시보드", style={'textAlign': 'center', 'marginBottom': '20px'}),

    html.Div(id='market-indices', style={'display': 'flex', 'justifyContent': 'center', 'gap': '20px', 'marginBottom': '20px'}),

    html.Div(style={'display': 'grid', 'gridTemplateColumns': '2fr 1fr', 'gap': '20px'}, children=[
        html.Div([
            dcc.Dropdown(id='ticker-selector', options=[{'label': f"{name} ({ticker})", 'value': ticker} for ticker, name in US_TOP10.items()], value='NVDA', style={'color': '#000', 'marginBottom': '10px'}),
            dcc.Graph(id='stock-chart', style={'height': '400px'}),
        ]),
        html.Div([
            # AI 예측 결과 출력 영역 추가
            html.Div(id='ai-prediction', style={'backgroundColor': '#151c2e', 'padding': '20px', 'borderRadius': '12px', 'marginBottom': '20px'}),
            html.Div(id='fear-greed-index', style={'backgroundColor': '#151c2e', 'padding': '20px', 'borderRadius': '12px', 'marginBottom': '20px'}),
            html.Div(id='news-feed', style={'backgroundColor': '#151c2e', 'padding': '20px', 'borderRadius': '12px', 'height': '300px', 'overflowY': 'auto'})
        ])
    ]),

    dcc.Interval(id='refresh-interval', interval=60000, n_intervals=0)
])

In [ ]:
# AI 예측 콜백 추가
@app.callback(
    Output('ai-prediction', 'children'),
    [Input('refresh-interval', 'n_intervals')]
)
def update_prediction(n):
    try:
        # "Processed_NASDAQ.csv" 파일에서 최근 60일치 데이터를 불러옵니다.
        # 실제 환경에서는 yfinance 라이브러리로 실시간 데이터를 불러와
        # CSV와 동일한 82개 컬럼 구조로 전처리하는 로직이 필요합니다.
        data = pd.read_csv("Processed_NASDAQ.csv", parse_dates=['Date'], index_col='Date')
        if 'Name' in data.columns:
            del data['Name']

        # 최근 60일 데이터 추출 및 3D 텐서 변환
        recent_60_days = data.tail(60).values
        scaled_data = scaler.transform(recent_60_days)
        X_input = np.array([scaled_data]) # shape: (1, 60, 82)

        # 상승 확률 예측
        pred_prob = model.predict(X_input)[0][0]
        prediction_text = "상승 예상" if pred_prob > 0.5 else "하락 예상"
        color = '#10b981' if pred_prob > 0.5 else '#ef4444'

        return [
            html.H3("🤖 AI 다음 날 종가 예측 (NASDAQ)"),
            html.Div(f"{prediction_text} (확률: {pred_prob*100:.2f}%)", style={'fontSize': '24px', 'fontWeight': 'bold', 'color': color})
        ]
    except Exception as e:
        return [html.Div(f"예측 데이터를 불러올 수 없습니다: {e}")]

In [ ]:
# 기존 콜백 유지
@app.callback(
    [Output('market-indices', 'children'), Output('fear-greed-index', 'children')],
    [Input('refresh-interval', 'n_intervals')]
)
def update_market_info(n):
    # 기존과 동일
    pass

@app.callback(
    Output('stock-chart', 'figure'),
    [Input('ticker-selector', 'value'), Input('refresh-interval', 'n_intervals')]
)
def update_chart(ticker, n):
    # 기존과 동일
    pass

@app.callback(
    Output('news-feed', 'children'),
    [Input('refresh-interval', 'n_intervals')]
)
def update_news(n):
    # 기존과 동일
    pass

if __name__ == '__main__':
    app.run(debug=True)